In [19]:
# Section 0: Import required libraries
from pathlib import Path
from collections import Counter
import numpy as np

# Step 2: Define dataset root directory
ROOT = Path("../data/lfw-deepfunneled/lfw-deepfunneled/")

# Step 3: Find all image files
img_paths = list(ROOT.rglob("*.jpg"))
print("Total images found:", len(img_paths))

# Step 4: Extract identity names (folder names) from image paths
ids = [p.parent.name for p in img_paths]
cnt = Counter(ids)
print("Unique identities:", len(cnt))
print("Top 10 identities by #images:", cnt.most_common(10))

# Step 5: Analyze distribution of image counts per identity
sizes = np.array(list(cnt.values()))
print(">=2 images identities:", (sizes >= 2).sum())
print(">=3 images identities:", (sizes >= 3).sum())
print(">=5 images identities:", (sizes >= 5).sum())

Total images found: 13233
Unique identities: 5749
Top 10 identities by #images: [('George_W_Bush', 530), ('Colin_Powell', 236), ('Tony_Blair', 144), ('Donald_Rumsfeld', 121), ('Gerhard_Schroeder', 109), ('Ariel_Sharon', 77), ('Hugo_Chavez', 71), ('Junichiro_Koizumi', 60), ('Jean_Chretien', 55), ('John_Ashcroft', 53)]
>=2 images identities: 1680
>=3 images identities: 901
>=5 images identities: 423


In [20]:
# Section 1:  Filter identities with at least m images
from collections import Counter, defaultdict

m = 2

keep_ids = {name for name, k in cnt.items() if k >= m}
print(f"Identities with >= {m} images:", len(keep_ids))

img_paths_keep = [p for p in img_paths if p.parent.name in keep_ids]
print("Images kept:", len(img_paths_keep))

# Step 7: Group images by identity
by_id = defaultdict(list)
for p in img_paths_keep:
    by_id[p.parent.name].append(p)
print("Min images per kept identity:", min(len(v) for v in by_id.values()))
print("Avg images per kept identity:",sum(len(v) for v in by_id.values()) / len(by_id))
print("Max images per kept identity:", max(len(v) for v in by_id.values()))

Identities with >= 2 images: 1680
Images kept: 9164
Min images per kept identity: 2
Avg images per kept identity: 5.454761904761905
Max images per kept identity: 530


In [21]:
# Section 2: Data Preparation for LFW Face Verification
import pandas as pd

DATA_DIR = Path("../data")
IMG_ROOT = ROOT

def lfw_img_path(name: str, idx: int) -> Path:
    """Construct LFW image path from name and index"""
    return IMG_ROOT / name / f"{name}_{idx:04d}.jpg"

def read_people_names(path: Path) -> list[str]:
    """Read identity names from CSV file"""
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]  # Remove extra whitespace in column names
    name_col = df.columns[0]
    return df[name_col].astype(str).tolist()

def read_pairs(path: Path, label: int) -> pd.DataFrame:
    """Read pair file (matched or mismatched)"""
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]  
    k = df.shape[1]

    # Case 1: Same-person pairs
    if k == 3:
        df = df.rename(columns={
            df.columns[0]: "name1",
            df.columns[1]: "idx1",
            df.columns[2]: "idx2"
        })
        df["name2"] = df["name1"]

    # Case 2: Different-person pairs
    elif k >= 4:
        df = df.rename(columns={
            df.columns[0]: "name1",
            df.columns[1]: "idx1",
            df.columns[2]: "name2",
            df.columns[3]: "idx2"
        })
    else:
        raise ValueError(f"Unexpected columns in {path}: {df.columns.tolist()}")

    df["idx1"] = pd.to_numeric(df["idx1"], errors="coerce").astype("Int64")
    df["idx2"] = pd.to_numeric(df["idx2"], errors="coerce").astype("Int64")
    df["label"] = label

    return df[["name1", "idx1", "name2", "idx2", "label"]]

# Step 1: Read training identities and intersect with keep_ids
train_people_path = DATA_DIR / "peopleDevTrain.csv"
train_names_raw = read_people_names(train_people_path)

train_ids = sorted(set(train_names_raw) & set(keep_ids))
print("Train identities (DevTrain ∩ keep_ids):", len(train_ids))

# Step 2: Collect all training image paths
train_img_paths = []
for name in train_ids:
    train_img_paths.extend(by_id[name])
print("Train images:", len(train_img_paths))

# Step 3: Read test pairs (same and different)
pairs_same = read_pairs(DATA_DIR / "matchpairsDevTest.csv", label=1)
pairs_diff = read_pairs(DATA_DIR / "mismatchpairsDevTest.csv", label=0)
test_pairs = pd.concat([pairs_same, pairs_diff], ignore_index=True)

# Step 4: Construct image paths for test pairs
test_pairs["path1"] = test_pairs.apply(lambda r: lfw_img_path(r["name1"], int(r["idx1"])), axis=1)
test_pairs["path2"] = test_pairs.apply(lambda r: lfw_img_path(r["name2"], int(r["idx2"])), axis=1)

# Step 5: Keep valid pairs
exists = (
    test_pairs["path1"].map(lambda p: p.exists()) &
    test_pairs["path2"].map(lambda p: p.exists())
)
test_pairs = test_pairs[exists].copy()

in_keep = (
    test_pairs["name1"].isin(keep_ids) &
    test_pairs["name2"].isin(keep_ids)
)
test_pairs = test_pairs[in_keep].copy()

print("Test pairs kept:", len(test_pairs))
print(test_pairs.head())

Train identities (DevTrain ∩ keep_ids): 1184
Train images: 6671
Test pairs kept: 541
              name1  idx1             name2  idx2  label  \
0      Abdullah_Gul    13      Abdullah_Gul    14      1   
1      Abdullah_Gul    13      Abdullah_Gul    16      1   
2  Abdullatif_Sener     1  Abdullatif_Sener     2      1   
3    Adel_Al-Jubeir     1    Adel_Al-Jubeir     3      1   
4         Al_Pacino     1         Al_Pacino     2      1   

                                               path1  \
0  ../data/lfw-deepfunneled/lfw-deepfunneled/Abdu...   
1  ../data/lfw-deepfunneled/lfw-deepfunneled/Abdu...   
2  ../data/lfw-deepfunneled/lfw-deepfunneled/Abdu...   
3  ../data/lfw-deepfunneled/lfw-deepfunneled/Adel...   
4  ../data/lfw-deepfunneled/lfw-deepfunneled/Al_P...   

                                               path2  
0  ../data/lfw-deepfunneled/lfw-deepfunneled/Abdu...  
1  ../data/lfw-deepfunneled/lfw-deepfunneled/Abdu...  
2  ../data/lfw-deepfunneled/lfw-deepfunneled/Abdu...

In [22]:
# Section 3: HOG Feature Extraction and PCA Dimensionality Reduction

from pathlib import Path
import numpy as np
import pandas as pd

from tqdm import tqdm 

from skimage.io import imread
from skimage.color import rgb2gray
from skimage.transform import resize
from skimage.feature import hog
from sklearn.decomposition import PCA

H, W = 128, 128

def load_gray_resized(img_path: Path, out_hw=(H, W)) -> np.ndarray:
    """Load image, convert to grayscale, resize to fixed resolution"""
    img = imread(str(img_path))
    if img.ndim == 3:
        img = rgb2gray(img)
    img = resize(img, out_hw, anti_aliasing=True, preserve_range=False)
    return img.astype(np.float32)

def hog_feat(img: np.ndarray) -> np.ndarray:
    """Extract HOG feature vector from image"""
    return hog(
        img,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm="L2-Hys",
        transform_sqrt=True,
        feature_vector=True,
    ).astype(np.float32)

def extract_hog_matrix(paths: list[Path], desc: str) -> np.ndarray:
    """Extract HOG features for a list of image paths"""
    X = []
    for p in tqdm(paths, desc=desc, total=len(paths), leave=True):
        g = load_gray_resized(p)
        X.append(hog_feat(g))
    return np.vstack(X)

# Step 1: Build training labels aligned with train_img_paths
train_labels = []
for name in train_ids:
    train_labels.extend([name] * len(by_id[name]))

assert len(train_labels) == len(train_img_paths), "train_labels must align with train_img_paths"

# Step 2: HOG + PCA on TRAINING images
CACHE_DIR = DATA_DIR / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
hog_train_cache = CACHE_DIR / "hog_train.npy"

if hog_train_cache.exists():
    X_train_hog = np.load(hog_train_cache)
else:
    X_train_hog = extract_hog_matrix(train_img_paths, desc="Extracting HOG (train)")
    np.save(hog_train_cache, X_train_hog)
print("Train HOG matrix:", X_train_hog.shape)

pca = PCA(n_components=50, svd_solver="randomized", whiten=False, random_state=0)
X_train_pca = pca.fit_transform(X_train_hog)
print("Train PCA matrix:", X_train_pca.shape)
print("Explained var ratio sum:", float(pca.explained_variance_ratio_.sum()))

# Step 3. Build clean training DataFrame
train_df = pd.DataFrame({
    "path": [str(p) for p in train_img_paths],
    "id": train_labels,
})
train_df["x_50d"] = list(X_train_pca.astype(np.float32))
print(train_df.head())
print("train_df shape:", train_df.shape)
print("Example x_50d shape:", train_df.loc[0, "x_50d"].shape)

# Step 4. Transform TEST PAIRS images using trained PCA
all_test_paths = (
    pd.concat([test_pairs["path1"], test_pairs["path2"]], ignore_index=True)
    .drop_duplicates()
    .tolist()
)
all_test_paths = [Path(p) for p in all_test_paths]

hog_test_cache = CACHE_DIR / "hog_test_unique.npy"
paths_test_cache = CACHE_DIR / "hog_test_unique_paths.csv"

def _build_test_cache():
    """Extract and cache HOG features for unique test images."""
    X = extract_hog_matrix(all_test_paths, desc="Extracting HOG (test unique)")
    np.save(hog_test_cache, X)
    pd.DataFrame({"path": list(map(str, all_test_paths))}).to_csv(
        paths_test_cache, index=False
    )
    return X, {str(p): i for i, p in enumerate(all_test_paths)}

if hog_test_cache.exists() and paths_test_cache.exists():
    saved = pd.read_csv(paths_test_cache)
    saved_paths = saved["path"].astype(str).tolist()

    if (len(saved_paths) == len(all_test_paths)) and \
       (saved_paths == list(map(str, all_test_paths))):
        X_test_hog_unique = np.load(hog_test_cache)
        path_to_row = {p: i for i, p in enumerate(saved_paths)}
    else:
        X_test_hog_unique, path_to_row = _build_test_cache()
else:
    X_test_hog_unique, path_to_row = _build_test_cache()

X_test_pca_unique = pca.transform(X_test_hog_unique).astype(np.float32)
print("Unique test imgs:", len(all_test_paths))
print("Unique test PCA:", X_test_pca_unique.shape)

def get_pca_vec(img_path: Path) -> np.ndarray:
    """Retrieve 50D PCA vector for a given test image path."""
    return X_test_pca_unique[path_to_row[str(img_path)]]

test_pairs["x1_50d"] = test_pairs["path1"].apply(get_pca_vec)
test_pairs["x2_50d"] = test_pairs["path2"].apply(get_pca_vec)
print(test_pairs[["label", "path1", "path2"]].head())
print("Example x1 shape:", test_pairs["x1_50d"].iloc[0].shape)


Train HOG matrix: (6671, 8100)
Train PCA matrix: (6671, 50)
Explained var ratio sum: 0.34002235531806946
                                                path             id  \
0  ../data/lfw-deepfunneled/lfw-deepfunneled/Aaro...  Aaron_Peirsol   
1  ../data/lfw-deepfunneled/lfw-deepfunneled/Aaro...  Aaron_Peirsol   
2  ../data/lfw-deepfunneled/lfw-deepfunneled/Aaro...  Aaron_Peirsol   
3  ../data/lfw-deepfunneled/lfw-deepfunneled/Aaro...  Aaron_Peirsol   
4  ../data/lfw-deepfunneled/lfw-deepfunneled/Aaro...   Aaron_Sorkin   

                                               x_50d  
0  [-2.181627, -1.7716255, -1.0017656, -0.6061927...  
1  [-0.42398614, -0.36349496, 2.8429947, -0.40662...  
2  [-1.4491417, -1.3662723, 0.32135573, -0.430760...  
3  [1.2979895, -0.6765771, -0.33226478, 1.6058495...  
4  [-0.65970355, -0.4217385, -0.45955947, 0.83454...  
train_df shape: (6671, 3)
Example x_50d shape: (50,)
Unique test imgs: 926
Unique test PCA: (926, 50)
   label                            

In [ ]:
# A1: Examine marginal distribution of each PCA dimension

import numpy as np
import pandas as pd
from scipy.stats import skew, kurtosis, normaltest

X = np.stack(train_df["x_50d"].to_numpy())  # (n,50)

stats = []
for j in range(X.shape[1]):
    xj = X[:, j]
    stats.append({
        "dim": j,
        "skew": float(skew(xj)),
        "kurtosis_excess": float(kurtosis(xj, fisher=True)),
        "normaltest_p": float(normaltest(xj).pvalue) 
    })
stats_df = pd.DataFrame(stats).sort_values("normaltest_p")
stats_df.head(10)

,dim,skew,kurtosis_excess,normaltest_p
0,0,0.149056,-0.750538,7.885305e-103
1,1,0.652777,0.034011,2.389947e-87
4,4,0.369604,0.773036,2.470530e-51
7,7,-0.261027,0.224042,3.237802e-19
11,11,0.172786,0.405009,6.685327e-15
13,13,0.216848,0.226638,1.959283e-14
25,25,0.038521,0.594130,3.347998e-14
9,9,0.189842,0.259336,1.406524e-12
22,22,-0.014282,0.459624,1.797424e-09
36,36,-0.157168,0.099639,3.293295e-07


In [24]:
# A2: Compare distance distributions (same vs different identity)
from numpy.linalg import norm

def vec_col_to_mat(s):
    return np.stack(s.to_numpy())

x1 = vec_col_to_mat(test_pairs["x1_50d"])
x2 = vec_col_to_mat(test_pairs["x2_50d"])
d = norm(x1 - x2, axis=1)
test_pairs = test_pairs.copy()
test_pairs["l2dist"] = d

test_pairs.groupby("label")["l2dist"].describe()

,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
0,41.0,8.260393,1.338261,5.989869,7.091528,8.066125,9.264488,11.337422
1,500.0,7.411689,1.270868,3.415161,6.669287,7.465687,8.306828,11.080956


In [25]:
# B1: Decompose total scatter into within-class and between-class
import numpy as np
import pandas as pd

train_df2 = train_df.copy()
train_df2["id"] = train_df2["id"].astype(str)

X = np.stack(train_df2["x_50d"].to_numpy())
ids = train_df2["id"].to_numpy()

mu = X.mean(axis=0, keepdims=True)

# within scatter
S_W = np.zeros((X.shape[1], X.shape[1]), dtype=np.float64)
# between scatter
S_B = np.zeros_like(S_W)

for pid, g in train_df2.groupby("id"):
    Xg = np.stack(g["x_50d"].to_numpy()).astype(np.float64)
    ng = Xg.shape[0]
    mug = Xg.mean(axis=0, keepdims=True)
    # within
    Xc = Xg - mug
    S_W += Xc.T @ Xc
    # between
    dm = (mug - mu)
    S_B += ng * (dm.T @ dm)

S_T = (X - mu).T @ (X - mu)

print("||S_T - (S_W+S_B)||_F =", np.linalg.norm(S_T - (S_W + S_B)))

ratio_between = np.trace(S_B) / np.trace(S_T)
ratio_within = np.trace(S_W) / np.trace(S_T)
print("trace ratio between:", float(ratio_between), "within:", float(ratio_within))

||S_T - (S_W+S_B)||_F = 0.004696796358704813
trace ratio between: 0.24143933776076903 within: 0.7585606915969493


In [26]:
# B2: Check homoscedasticity (equal covariance) assumption
import numpy as np
import pandas as pd

within_var = []
for pid, g in train_df.groupby("id"):
    Xg = np.stack(g["x_50d"].to_numpy())
    if Xg.shape[0] < 2:
        continue
    covg = np.cov(Xg, rowvar=False)
    within_var.append({
        "id": pid,
        "n": Xg.shape[0],
        "trace_cov": float(np.trace(covg)),
        "mean_var": float(np.mean(np.diag(covg))),
    })

within_df = pd.DataFrame(within_var)
within_df[["trace_cov","mean_var"]].describe(percentiles=[.05,.25,.5,.75,.95])

,trace_cov,mean_var
count,1184.000000,1184.000000
mean,28.536486,0.570730
std,8.236575,0.164732
min,0.960915,0.019218
5%,15.373337,0.307467
25%,22.863614,0.457272
50%,28.590644,0.571813
75%,33.703426,0.674069
95%,42.260932,0.845219
max,62.956776,1.259136


In [27]:
# B3: Check Gaussian generative assumption through whitening

import numpy as np
from scipy.linalg import cho_factor, cho_solve

X = np.stack(train_df["x_50d"].to_numpy()).astype(np.float64)
Xc = X - X.mean(axis=0, keepdims=True)
Sigma = np.cov(Xc, rowvar=False)

ridge = 1e-5
Sigma_r = Sigma + ridge*np.eye(Sigma.shape[0])

# Compute squared Mahalanobis distances
c, low = cho_factor(Sigma_r, lower=True, check_finite=False)
r2 = np.einsum("ij,ij->i", Xc, cho_solve((c, low), Xc.T, check_finite=False).T)

print("mean r2:", float(r2.mean()), "theory:", Sigma.shape[0])
print("var  r2:", float(r2.var()),  "theory:", 2*Sigma.shape[0])

mean r2: 49.99136067029697 theory: 50
var  r2: 138.55982044807612 theory: 100
